# Task 2.1 — Linear Regression (Baseline)

In [2]:
# ============================================================
# SETUP - run this first, before any task below
# ============================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('clean_dataset.csv')

X = df.drop(columns=['monthly_salary'])
y = df['monthly_salary']

# Same 80/20 split as Day 1, Task 1.1 (same random_state for consistency)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Always start with the simplest model as your baseline
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print('Linear Regression Results:')
print(f'  MAE:  {mean_absolute_error(y_test, y_pred):.2f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')
print(f'  R2:   {r2_score(y_test, y_pred):.3f}')

# Look at coefficients to understand feature importance
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)
print(coef_df.head(10))

Linear Regression Results:
  MAE:  0.26
  RMSE: 0.36
  R2:   0.670
                   feature  coefficient
10     position_SUPERVISOR     0.869050
8        position_ENGINEER     0.677160
11     position_TECHNICIAN    -0.647616
6           position_CLERK    -0.283872
5   position_ADMINISTRATOR    -0.264545
15           department_HR    -0.264545
9         position_LABORER    -0.260854
16           department_IT     0.178845
4      position_ACCOUNTANT    -0.110302
0              punch_count     0.085166


# Task 2.2 — Ridge & Lasso

In [4]:
from sklearn.linear_model import Ridge, Lasso

# Ridge (L2): shrinks coefficients, keeps all features
# Lasso (L1): can shrink coefficients to exactly zero (feature selection)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print(f'Ridge R2: {r2_score(y_test, y_pred_ridge):.3f}')

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print(f'Lasso R2: {r2_score(y_test, y_pred_lasso):.3f}')

# Check how many features Lasso zeroed out
n_zero = (lasso.coef_ == 0).sum()
print(f'Lasso zeroed out {n_zero} of {len(lasso.coef_)} features')
# L2 (Ridge) shrinks coefficients smoothly, keeps all features.
# L1 (Lasso) can push weak features to exactly 0 -> automatic feature selection.

Ridge R2: 0.681
Lasso R2: 0.070
Lasso zeroed out 16 of 18 features


# Task 2.3 — Tree-Based Regressors

In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Decision Tree - simple, interpretable, prone to overfitting
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f'Decision Tree R2: {r2_score(y_test, y_pred_dt):.3f}')

# Random Forest - ensemble of trees, usually more accurate and stable
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f'Random Forest R2: {r2_score(y_test, y_pred_rf):.3f}')

# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df.head(10))

Decision Tree R2: 0.565
Random Forest R2: -2.895
                   feature  importance
1             hours_worked    0.378364
0              punch_count    0.152884
10     position_SUPERVISOR    0.132473
8        position_ENGINEER    0.129965
2       satisfaction_score    0.064069
7          position_DRIVER    0.063406
16           department_IT    0.029682
11     position_TECHNICIAN    0.020913
5   position_ADMINISTRATOR    0.006922
15           department_HR    0.006021


# Task 2.4 — Compare All Regression Models

In [7]:
results = {
    'Linear Regression': r2_score(y_test, y_pred),
    'Ridge': r2_score(y_test, y_pred_ridge),
    'Lasso': r2_score(y_test, y_pred_lasso),
    'Decision Tree': r2_score(y_test, y_pred_dt),
    'Random Forest': r2_score(y_test, y_pred_rf),
}

results_df = pd.DataFrame(results.items(), columns=['Model', 'R2 Score'])
results_df = results_df.sort_values('R2 Score', ascending=False)
print(results_df)

results_df.to_csv('regression_comparison.csv', index=False)

               Model  R2 Score
1              Ridge  0.680927
0  Linear Regression  0.669762
3      Decision Tree  0.564723
2              Lasso  0.069585
4      Random Forest -2.895398
